# Thesis Note — Step 1
# Initial Multimodal Dataset Audit
## Purpose

The first stage of the study aimed to understand the overall structure of the COde dataset before designing the machine learning pipeline. Rather than directly training models, a comprehensive audit was conducted to characterize the availability of each modality and identify potential limitations that could affect downstream experiments.

## Analysis Performed

The audit examined:

number of patients
number of visits
availability of clinical text
availability of intraoral photographs
availability of radiographs
multimodal completeness at visit level
modality availability at patient level

## Main Findings

The audit revealed that:

the dataset contains 4,800 unique patients and 8,775 clinical visits.
clinical text is available for nearly every visit.
photographs are available for almost every visit.
radiographs are naturally missing for a substantial portion of visits.
only approximately half of the visits contain all three modalities simultaneously.

## Conclusion

The missing radiographs are an intrinsic property of the dataset rather than a preprocessing artifact. Consequently, the dataset naturally supports missing-modality research scenarios.

# Thesis Note — Step 2
# Cross-Visit Duplication Audit
## Purpose

After understanding modality availability, the next objective was to determine whether images were reused across multiple visits belonging to the same patient.

Such reuse could introduce information leakage if patient visits were separated across training and testing subsets.

## Analysis Performed

The audit analyzed:

duplicated image filenames
duplicated visit pairs
duplicated patient-modality pairs
consecutive versus non-consecutive reuse
treatment progression
temporal relationships between visits
## Main Findings

The analysis showed that image reuse is relatively common.

Important observations include:

826 patients exhibit image reuse across multiple visits.
4,959 unique images are reused.
980 unique visit pairs contain duplicated images.
914 duplicated visit pairs occur in consecutive visits.
only 66 duplicated visit pairs occur across non-consecutive visits.
image reuse usually corresponds to longitudinal follow-up rather than accidental duplication.
## Conclusion

Image reuse represents clinically meaningful longitudinal follow-up rather than annotation errors. Therefore, preventing leakage requires splitting data at the patient level rather than the visit level.

# Thesis Note — Step 3
# Evaluation of the Original Dataset Splits
## Purpose

Before constructing a new data split, the official dataset partitions were examined to determine whether they satisfy the requirements of patient-level evaluation.

## Analysis Performed

The following official files were analyzed:

train.json
train-cls.json
train-diagnostic.json
test_cls.json
test_diagnostic.json

For each split, patient identifiers, visit identifiers, and image identifiers were extracted.

Pairwise overlap was then computed between all training and testing partitions.

## Main Findings

The analysis revealed that:

no visit is shared between train and test.
no image filename is shared between train and test.
however, 245 patients appear in both training and testing partitions.

This observation is particularly important because patients may have multiple visits over time.

Although duplicated images often receive different filenames in later visits, they still represent the same underlying clinical examination.

Therefore, absence of filename overlap does not guarantee absence of information leakage.

## Conclusion

The official split is unsuitable for patient-level evaluation.

A completely new patient-level split must therefore be generated before any experimental evaluation.

# Thesis Note — Step 4
# Definition of the Patient-Level Split Policy
## Purpose

Before implementing a new splitting algorithm, a formal splitting policy was defined to ensure that every subsequent experiment follows an identical evaluation protocol.

## Patient-Level Splitting Principle

The fundamental splitting unit is the patient rather than the visit or individual image.

Each patient must belong exclusively to one partition.

Therefore,

no patient may appear in multiple splits,
all visits of a patient remain together,
all associated photographs remain together,
all associated radiographs remain together.
## Proposed Policy

The study adopts the following principles:

patient-level splitting
reproducible random seed
stratification according to diagnosis distribution
preservation of radiograph availability distribution
preservation of longitudinal visit structure
leakage verification after split generation
## Expected Outcome

This policy guarantees unbiased evaluation while preserving the natural multimodal characteristics of the dataset.

# Thesis Note — Step 5
# Design of the Patient-Level Split Pipeline
## Purpose

After defining the splitting policy, the next step is to design a reproducible pipeline capable of generating patient-level splits automatically.

## Pipeline Responsibilities

The proposed pipeline will:

load the complete dataset,
identify unique patients,
construct patient-level train, validation, and test partitions,
assign every visit to its corresponding patient split,
perform leakage verification,
verify modality distributions,
verify diagnosis distributions,
verify radiograph availability,
verify duplicated-image isolation,
generate complete audit reports.
## Pipeline Outputs

The pipeline will generate:

patient-to-split mapping
visit-level split assignments
summary statistics
leakage reports
distribution comparison reports
reproducible configuration files
## Expected Contribution

The pipeline forms the foundation of all subsequent self-supervised pretraining and downstream multimodal experiments.

By enforcing patient-level isolation, every experimental result reported in this thesis will satisfy rigorous evaluation standards and eliminate patient-level information leakage.

# Thesis Note — Step 6
# Implementation of the Patient-Level Split Pipeline

## Purpose

Following the finalized patient-level splitting policy, this step implements a reproducible pipeline for generating train, validation, and test partitions at the patient level.

The main objective is to create a reliable dataset partitioning mechanism that prevents patient-level mixing across experimental subsets and provides consistent split artifacts for all subsequent experiments.

---

## Implementation Details

The implemented pipeline operates on the complete COde dataset and performs the following steps:

- loads the original dataset,
- removes the original dataset split assignments to avoid dependency on the provided split,
- identifies unique patients,
- constructs patient-level metadata,
- creates patient-level stratification indicators,
- generates train, validation, and test partitions,
- assigns every visit to the split of its corresponding patient,
- saves reproducible split artifacts.

---

## Patient-Level Split Strategy

The split unit is defined as:
patient_id

Therefore, all visits belonging to the same patient are always assigned to the same subset.

The implemented ratios are:

| Split | Ratio |
|---|---|
| Train | 70% |
| Validation | 15% |
| Test | 15% |

The random seed is fixed:
seed = 42


to guarantee reproducibility.

---

## Stratification Strategy

Due to the highly heterogeneous nature of the diagnosis annotations, raw diagnosis labels were not used directly for stratification.

The dataset contains:

- 3422 unique diagnosis values,
- free-text diagnostic descriptions,
- multiple variations of clinically similar conditions.

Therefore, patient-level availability indicators were used instead.

The stratification features are:

1. Diagnosis availability

Whether the patient has at least one visit with available diagnosis information.

2. Radiograph availability

Whether the patient has at least one visit containing radiographic information.

This creates four clinically relevant groups:

| Diagnosis | Radiograph | Group |
|---|---|---|
| Available | Available | True_True |
| Available | Missing | True_False |
| Missing | Available | False_True |
| Missing | Missing | False_False |

---

## Generated Artifacts

The pipeline generates the following outputs:
results/patient_level_split/

├── patient_split.csv
├── visit_split.csv
└── split_summary.json


### patient_split.csv

Contains the final patient-level mapping:
patient_id → split



### visit_split.csv

Contains the original dataset records with the newly assigned patient-level split.

### split_summary.json

Stores reproducible information about:

- random seed,
- split ratios,
- number of patients per subset,
- number of visits per subset.

---

## Implementation Result

The generated split contains:

| Split | Patients | Visits |
|---|---:|---:|
| Train | 3360 | 6129 |
| Validation | 720 | 1330 |
| Test | 720 | 1316 |

The total dataset coverage remains:

- 4800 unique patients
- 8775 visits

---

## Separation From Subsequent Audits

This implementation step only generates the patient-level split artifacts.

The following validations are intentionally separated into independent milestones:

- Leakage Validation
- Split Quality Audit

These steps will verify:

- patient isolation,
- visit isolation,
- image isolation,
- modality distribution consistency,
- diagnostic distribution consistency.

---

## Expected Contribution

This pipeline establishes the foundation for all subsequent self-supervised pretraining and multimodal experiments.

By enforcing patient-level isolation at the dataset partitioning stage, future experiments can be conducted under a reproducible evaluation protocol with reduced risk of patient-level information leakage.

# Thesis Note — Step 7
# Leakage Validation of the Patient-Level Split

## Purpose

After generating the patient-level train, validation, and test partitions, an independent leakage validation procedure was performed to verify the integrity of the dataset split.

The objective of this step was to ensure that no patient, visit, or image reference information was shared between different experimental subsets.

---

## Validation Scope

The leakage validation pipeline evaluated the following potential sources of information leakage:

- patient-level overlap,
- visit-level overlap,
- photograph filename overlap,
- radiograph filename overlap,
- inconsistent split assignments.

The validation was performed using the generated patient-level split artifacts without modifying the original dataset or split assignments.

---

## Assignment Consistency Validation

The pipeline first verified that every entity was assigned to exactly one split.

Validated entities:

- patient_id
- checkup_id

The validation results were:

| Entity | Total | Multiple Assignments |
|---|---:|---:|
| Patients | 4800 | 0 |
| Visits | 8775 | 0 |

The assignment validation status was:
# Thesis Note — Step 7
# Leakage Validation of the Patient-Level Split

## Purpose

After generating the patient-level train, validation, and test partitions, an independent leakage validation procedure was performed to verify the integrity of the dataset split.

The objective of this step was to ensure that no patient, visit, or image reference information was shared between different experimental subsets.

---

## Validation Scope

The leakage validation pipeline evaluated the following potential sources of information leakage:

- patient-level overlap,
- visit-level overlap,
- photograph filename overlap,
- radiograph filename overlap,
- inconsistent split assignments.

The validation was performed using the generated patient-level split artifacts without modifying the original dataset or split assignments.

---

## Assignment Consistency Validation

The pipeline first verified that every entity was assigned to exactly one split.

Validated entities:

- patient_id
- checkup_id

The validation results were:

| Entity | Total | Multiple Assignments |
|---|---:|---:|
| Patients | 4800 | 0 |
| Visits | 8775 | 0 |

The assignment validation status was:
# Thesis Note — Step 7
# Leakage Validation of the Patient-Level Split

## Purpose

After generating the patient-level train, validation, and test partitions, an independent leakage validation procedure was performed to verify the integrity of the dataset split.

The objective of this step was to ensure that no patient, visit, or image reference information was shared between different experimental subsets.

---

## Validation Scope

The leakage validation pipeline evaluated the following potential sources of information leakage:

- patient-level overlap,
- visit-level overlap,
- photograph filename overlap,
- radiograph filename overlap,
- inconsistent split assignments.

The validation was performed using the generated patient-level split artifacts without modifying the original dataset or split assignments.

---

## Assignment Consistency Validation

The pipeline first verified that every entity was assigned to exactly one split.

Validated entities:

- patient_id
- checkup_id

The validation results were:

| Entity | Total | Multiple Assignments |
|---|---:|---:|
| Patients | 4800 | 0 |
| Visits | 8775 | 0 |

The assignment validation status was:
PASS


---

## Patient-Level Leakage Validation

The patient identifiers were compared pairwise between:

- Train and Validation
- Train and Test
- Validation and Test

The result was:
Shared patients = 0


This confirms that no patient information is distributed across multiple subsets.

---

## Visit-Level Leakage Validation

The same procedure was applied to visit identifiers (`checkup_id`).

The result was:
Shared visits = 0


Therefore, longitudinal records from the same visit are fully isolated between subsets.

---

## Image Reference Leakage Validation

Image filename references were extracted from:

- photographs
- radiographs

Each modality was independently checked for cross-split overlap.

Results:

| Modality | Shared Image References |
|---|---:|
| Photographs | 0 |
| Radiographs | 0 |

No identical image references were found across train, validation, and test partitions.

---

## Validation Results Summary

Final leakage validation summary:

| Validation Type | Result |
|---|---:|
| Patient overlap | 0 |
| Visit overlap | 0 |
| Photograph overlap | 0 |
| Radiograph overlap | 0 |
| Assignment errors | 0 |

Overall status:
SAFE


---

## Interpretation

The leakage validation confirms that the generated patient-level split successfully isolates all patients and their corresponding clinical records across experimental subsets.

This prevents direct information transfer between training and evaluation data caused by:

- repeated patient visits,
- longitudinal records,
- reused image references,
- shared clinical samples.

---

## Separation From Further Analysis

This step validates structural dataset isolation only.

The following analyses are performed separately in the next milestone:

- split quality assessment,
- modality distribution comparison,
- diagnosis distribution analysis,
- missing radiograph distribution analysis,
- duplicate image content analysis.

---

## Expected Contribution

By independently validating the generated patient-level split, all future self-supervised pretraining and multimodal downstream experiments are performed under a controlled evaluation setting.

This ensures that reported performance metrics represent true generalization to unseen patients rather than memorization of patient-specific information.